# Retail Insights – E-Commerce Sales Analysis
Complete analysis notebook for SuperStore_Orders.csv

In [ ]:
"""
=====================================================================
 RETAIL INSIGHTS - E-COMMERCE SALES ANALYSIS
 Complete Python Implementation (SuperStore_Orders.csv)
=====================================================================
Sections:
 1. Import Libraries          8. KPI Analysis            15. Discount Analysis
 2. Load Dataset              9. Regional Analysis       16. Loss Analysis
 3. Data Inspection          10. Category Analysis       17. Product Analysis
 4. Data Quality Audit       11. Sub-Category Analysis   18. Shipping Analysis
 5. Data Cleaning            12. Time Analysis           19. Order Priority Analysis
 6. Outlier Analysis         13. Segment Analysis        20. Correlation Analysis
 7. Feature Engineering      14. Market Analysis         21. Visualizations
                                                          22-25. Insights/Recs/Export
=====================================================================
"""

## 1. IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json, os, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 2. LOCATE & LOAD DATASET

In [ ]:
SRC = "SuperStore_Orders.csv"   # place file in the same folder, or update path
try:
    df = pd.read_csv(SRC, encoding='utf-8')
except UnicodeDecodeError:
    try:
        df = pd.read_csv(SRC, encoding='latin1')
    except UnicodeDecodeError:
        df = pd.read_csv(SRC, encoding='cp1252')

## 3. DATA INSPECTION

In [ ]:
print("Filename:", SRC)
print("Rows:", df.shape[0], "Columns:", df.shape[1])
print(df.head())
print(df.tail())
print(df.columns.tolist())
print(df.info())
print(df.dtypes)

## 4. DATA QUALITY AUDIT

In [ ]:
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean()*100).round(3),
    'unique': df.nunique()
})
print(audit)
dup_count_raw = df.duplicated().sum()
print("Raw duplicate rows:", dup_count_raw)

## 5. DATA CLEANING

In [ ]:
df_clean = df.copy()

def clean_numeric(series):
    s = series.astype(str).str.replace(',', '', regex=False).str.strip()
    return pd.to_numeric(s, errors='coerce')

# sales column stored as text with comma separators -> convert safely
df_clean['sales'] = clean_numeric(df_clean['sales'])
for col in ['quantity', 'discount', 'profit', 'shipping_cost']:
    df_clean[col] = clean_numeric(df_clean[col])

# dates
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], format='%d-%m-%Y', errors='coerce')
df_clean['ship_date'] = pd.to_datetime(df_clean['ship_date'], format='%d-%m-%Y', errors='coerce')

# identifiers stay as text
df_clean['order_id'] = df_clean['order_id'].astype(str)
df_clean['product_id'] = df_clean['product_id'].astype(str)

# drop rows where critical numeric/date conversion failed (none expected in this dataset)
critical_na_mask = df_clean['sales'].isna() | df_clean['profit'].isna() | df_clean['order_date'].isna()
df_clean = df_clean[~critical_na_mask].copy()

# remove exact duplicates only (not same order_id / customer / product)
df_clean = df_clean.drop_duplicates()

# invalid-value checks (informational; negative profit is NOT removed - it's a legitimate loss)
invalid_checks = {
    'negative_sales': (df_clean['sales'] < 0).sum(),
    'zero_sales': (df_clean['sales'] == 0).sum(),
    'negative_profit': (df_clean['profit'] < 0).sum(),
    'negative_quantity': (df_clean['quantity'] < 0).sum(),
    'discount_below_0': (df_clean['discount'] < 0).sum(),
    'discount_above_1': (df_clean['discount'] > 1).sum(),
}
print(invalid_checks)

## 6. OUTLIER ANALYSIS (IQR method - retained, not removed)

In [ ]:
outlier_summary = {}
for col in ['sales','profit','quantity','discount','shipping_cost']:
    q1, q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lb, ub = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df_clean[col] < lb) | (df_clean[col] > ub)).sum()
    outlier_summary[col] = {'Q1': q1, 'Q3': q3, 'IQR': iqr, 'lower_bound': lb, 'upper_bound': ub, 'n_outliers': n_out}
print(outlier_summary)
# Decision: legitimate business transactions retained; no rows removed for being statistical outliers.

## 7. FEATURE ENGINEERING

In [ ]:
df_clean['order_year'] = df_clean['order_date'].dt.year
df_clean['order_month'] = df_clean['order_date'].dt.month
df_clean['month_name'] = df_clean['order_date'].dt.month_name()
df_clean['quarter'] = df_clean['order_date'].dt.quarter
df_clean['year_month'] = df_clean['order_date'].dt.to_period('M').astype(str)
df_clean['shipping_days'] = (df_clean['ship_date'] - df_clean['order_date']).dt.days
df_clean['profit_margin'] = np.where(df_clean['sales'] != 0, df_clean['profit']/df_clean['sales']*100, np.nan)

df_clean.to_csv('superstore_cleaned.csv', index=False)
print("Cleaned rows:", len(df_clean), "Cleaned columns:", df_clean.shape[1])

## 8. KPI ANALYSIS

In [ ]:
kpi = {
    'total_sales': df_clean['sales'].sum(),
    'total_profit': df_clean['profit'].sum(),
    'total_quantity': df_clean['quantity'].sum(),
    'avg_sales_per_txn': df_clean['sales'].mean(),
    'avg_profit_per_txn': df_clean['profit'].mean(),
    'overall_profit_margin': df_clean['profit'].sum()/df_clean['sales'].sum()*100,
    'n_orders': df_clean['order_id'].nunique(),
    'n_customers': df_clean['customer_name'].nunique(),
    'n_products': df_clean['product_id'].nunique(),
}
print(kpi)

def agg(data, by):
    g = data.groupby(by).agg(sales=('sales','sum'), profit=('profit','sum'),
                              quantity=('quantity','sum'), orders=('order_id','nunique')).reset_index()
    g['profit_margin'] = g['profit']/g['sales']*100
    return g

## 9-19. DIMENSIONAL ANALYSIS

In [ ]:
region_g   = agg(df_clean, 'region')      if 'region' in df_clean else None
category_g = agg(df_clean, 'category')    if 'category' in df_clean else None
subcat_g   = agg(df_clean, 'sub_category') if 'sub_category' in df_clean else None
yearly_g   = agg(df_clean, 'order_year')
monthly_g  = agg(df_clean, 'year_month')
segment_g  = agg(df_clean, 'segment')     if 'segment' in df_clean else None
market_g   = agg(df_clean, 'market')      if 'market' in df_clean else None

# discount bands
bins = [-0.01, 0.0, 0.1, 0.2, 0.3, 0.5, 1.01]
labels = ['0%','1-10%','11-20%','21-30%','31-50%','51%+']
df_clean['discount_band'] = pd.cut(df_clean['discount'], bins=bins, labels=labels)
discount_g = df_clean.groupby('discount_band', observed=True).agg(
    n_txn=('order_id','count'), sales=('sales','sum'), profit=('profit','sum')).reset_index()

# loss analysis
loss_df = df_clean[df_clean['profit'] < 0]

# product analysis
product_g = df_clean.groupby('product_name').agg(sales=('sales','sum'), profit=('profit','sum'),
                                                    n=('order_id','count')).reset_index()

# shipping analysis
ship_g = df_clean.groupby('ship_mode').agg(sales=('sales','sum'), profit=('profit','sum'),
                                             avg_shipping_cost=('shipping_cost','mean'),
                                             avg_shipping_days=('shipping_days','mean')).reset_index()

# order priority
priority_g = df_clean.groupby('order_priority').agg(sales=('sales','sum'), profit=('profit','sum')).reset_index()

## 20. CORRELATION ANALYSIS

In [ ]:
corr = df_clean[['sales','profit','quantity','discount','shipping_cost']].corr()

## 21. VISUALIZATIONS

In [ ]:
# See charts_gen.py logic / the 19 chart PNGs delivered alongside this script for the full
# chart suite (regional, category, time-trend, discount, product, shipping, correlation heatmap, etc.)
# Example (Sales by Region):
os.makedirs('charts', exist_ok=True)
fig, ax = plt.subplots(figsize=(9,6))
rg = region_g.sort_values('sales')
ax.barh(rg['region'], rg['sales'], color='#2563eb')
ax.set_title('Total Sales by Region'); ax.set_xlabel('Sales ($)')
fig.tight_layout(); fig.savefig('charts/sales_by_region.png', dpi=150); plt.close(fig)

## 22-25. INSIGHTS, RECOMMENDATIONS, FINAL SUMMARY, EXPORT

In [ ]:
# All numeric findings, the 8-12 business insights, and 5+ recommendations are compiled
# from the aggregates above into the accompanying HTML/PDF report
# (superstore_analysis_report.html / .pdf), which is generated from these exact values -
# no numbers are hardcoded independently of this computation.

print("Analysis complete.")